## Init

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

from easyner.database.sqlite_backend.db_main import EasyNerDBHandler

db = EasyNerDBHandler()

In [ ]:
db.optimize_db_performance_parameters()

In [ ]:
db.cache_manager.clear_all()

## Debug

In [ ]:
db.statistics.debug_database_schema()

In [ ]:
%autoreload 2
print(db.statistics.total_valid_named_entities)
db.statistics.named_entities_count()


In [ ]:
stats = db.statistics

stats.documents_with_entities()

In [ ]:
stats.documents_with_entities(included_ne_classes=["DIS"])

In [ ]:
stats.documents_with_entities(included_ne_classes=["PNM"])

In [ ]:
stats.documents_with_entities(included_ne_classes=["DIS", "PNM"])

In [ ]:

stats.documents_with_entities(included_ne_classes=["DIS"], excluded_ne_classes=["PNM"])


## Database stats

In [ ]:
# Get schema from database
schema = db.schema
print(schema)

In [ ]:
doc_count = db.statistics.document_count
sent_count = db.statistics.sentence_count
entity_count = db.statistics.named_entities_count()



In [ ]:
# Make df from doc_count, sent_count, entity_count
import pandas as pd

df = pd.DataFrame({
    "doc_count": [doc_count],
    "sent_count": [sent_count],
    "entity_count": [entity_count],
})
df

In [ ]:
# Show as flowchart from left to right
from graphviz import Digraph
from IPython.display import display


def create_flowchart():
    dot = Digraph()

    # Add nodes
    dot.node('A', 'Documents')
    dot.node('B', 'Sentences')
    dot.node('C', 'Named Entities')

    # Add edges
    dot.edge('A', 'B')
    dot.edge('B', 'C')

    return dot
flowchart = create_flowchart()
display(flowchart)

In [ ]:
db.statistics.info_compression

```markdown
## Pointwise Mutual Information (PMI)

The Pointwise Mutual Information (PMI) measures the association between two events. In the context of text analysis, it's often used to quantify the relationship between two words or terms. The PMI between two words, $x$ and $y$, is defined as:

$$
PMI(x, y) = \log_2 \frac{p(x, y)}{p(x)p(y)}
$$

Where:

-   $p(x, y)$ is the joint probability of words $x$ and $y$ occurring together.
-   $p(x)$ is the probability of word $x$ occurring.
-   $p(y)$ is the probability of word $y$ occurring.

In practice, these probabilities are often estimated from corpus frequencies:

-   $p(x, y) = \frac{count(x, y)}{N}$
-   $p(x) = \frac{count(x)}{N}$
-   $p(y) = \frac{count(y)}{N}$

Where:

-   $count(x, y)$ is the number of times $x$ and $y$ occur together in a specific context (e.g., within a window of words).
-   $count(x)$ is the number of times $x$ occurs.
-   $count(y)$ is the number of times $y$ occurs.
-   $N$ is the total number of observations (e.g., total number of word pairs).

Thus, the formula can be rewritten as:

$$
PMI(x, y) = \log_2 \frac{count(x, y) \cdot N}{count(x) \cdot count(y)}
$$
```

## Entity Document Distribution

In [ ]:
%autoreload 2

from easyner.database.sqlite_backend.statistics.ne_doc_distr import Flowchart

flowchart = Flowchart(db.statistics)
data = flowchart.data

In [ ]:
flowchart.data

$$
\\begin{tabular}{llrr}\n & category & count & percentage \\\\\n0 & Total Documents & 16683210 & 100.000000 \\\\\n1 & Documents with Named Entities & 9936043 & 59.560000 \\\\\n2 & Documents without Named Entities & 6747167 & 40.440000 \\\\\n3 & Documents with DIS Entities & 9810448 & 58.800000 \\\\\n4 & Documents with PNM Entities & 169639 & 1.020000 \\\\\n5 & Documents with both DIS and PNM & 44044 & 0.260000 \\\\\n6 & Documents with DIS only & 9766404 & 58.540000 \\\\\n7 & Documents with PNM only & 125595 & 0.750000 \\\\\n\\end{tabular}\n

$$

$$

In [ ]:
flowchart.create_sankey_layers_dataframe()

flowchart.

In [ ]:
flowchart.get_sankey_data()

In [ ]:
%autoreload 2

# Import the Flowchart class from your project structure

fig = flowchart.render_sankey_diagram()
# fig is html, render it in a notebook cell
fig.show()

In [ ]:
for x_coordinate, column_name in enumerate(["Total Documents","Document entities", "Entity Distribution"]):
    fig.add_annotation(
          x=x_coordinate,#Plotly recognizes 0-5 to be the x range.

          y=1.075,#y value above 1 means above all nodes
          xref="x",
          yref="paper",
          text=column_name,#Text
          showarrow=False,
          font=dict(
              family="Tahoma",
              size=16,
              color="black",
              ),
          align="left",
          )
fig.show()

In [ ]:
import pandas as pd

df.columns = pd.MultiIndex.from_tuples([
    ("Numeric", "Integers"),
    ("Numeric", "Floats"),
    ("Non-Numeric", "Strings"),
])
df.index = pd.MultiIndex.from_tuples([
    ("L0", "ix1"), ("L0", "ix2"), ("L1", "ix3"),
])
s = df.style.highlight_max(
    props='cellcolor:[HTML]{FFFF00}; color:{red}; itshape:; bfseries:;',
)
s.to_latex(
    column_format="rrrrr", position="h", position_float="centering",
    hrules=True, label="table:5", caption="Styled LaTeX Table",
    multirow_align="t", multicol_align="r",
)

## Color schema

In [ ]:
# Define color scheme as class attribute for reuse across visualization methods
COLOR_SCHEME = {
    # Node colors
    'nodes': {
        'total_documents': "hsl(0, 5%, 76%)",          # Light gray for Total Documents
        'with_entities': "hsl(171, 18%, 63%)",         # Tan/gold for With Named Entities
        'without_entities': "hsl(142, 6%, 35%)",       # Green for Without Named Entities
        'disly': "hsl(12, 48%, 43%)",               # Intense pink/red for DIS Only
        'pnm_only': "hsl(38, 100%, 68%)",              # Light green for PNM Only
        'both_dis_pnm': "hsl(230, 55%, 65%)",          # Mixed color for Both
        'default': "rgba(150, 150, 150, 0.8)",          # Default gray
    },
    # Link colors
    'links': {
        'total_to_with_entities': "hsl(171, 18%, 63%)", # Matches with_entities node
        'total_to_without_entities': "hsl(351, 97%, 85%)", # Per your request
        'with_entities_to_dis_only': "hsl(16, 41%, 58%)", # Light version of DIS
        'with_entities_to_both': "rgba(140, 150, 210, 0.4)", # Matching the mixed color
        'with_entities_to_pnm_only': "hsl(44, 60%, 58%)", # Light version of PNM
        'total_to_pnm_only': "hsl(145, 7%, 78%)",       # Special case
        'default': "hsl(171, 17%, 60%)",                 # Default light gray
    },
}




In [ ]:
from easyner.database.sqlite_backend.statistics.color_scheme import *

## Entity Error percentage bar plot

### Debug

In [ ]:
import pandas as pd

errors: pd.DataFrame = db.statistics.count_named_entity_errors()


In [ ]:
errors
error_df = errors

In [ ]:
pivot_df = error_df.pivot(
            index="named_entity_class", columns="error_id", values="error_count",
        ).fillna(0).astype(int)

pivot_df

In [ ]:
entity_stats = db.statistics.results_entity_occurrence_errors()
entity_stats

In [ ]:
error_df = db.statistics.count_named_entity_errors()

# Fetch fq data from named_entities table
dis_fq = db.statistics.named_entities_count("DIS")
pnm_fq = db.statistics.named_entities_count("PNM")

fq_data = [
    ("DIS", dis_fq),
    ("PNM", pnm_fq),
]


In [ ]:
fq_data

In [ ]:
error_df

In [ ]:
pivot_df = error_df.pivot(index="named_entity_class", columns="error_id", values="error_count").fillna(0).astype(int)
pivot_df

In [ ]:
result_df = entity_stats[['named_entity_class', 'AMBIG', 'MISLAB']].copy()
result_df.set_index('named_entity_class', inplace=True)
result_df['total_errors'] = result_df.sum(axis=1)
result_df

In [ ]:
pnm_fq = db.statistics.named_entities_count("PNM")
pnm_fq

### Run

In [ ]:
# Get data
entity_stats = db.statistics.results_entity_occurrence_errors()

# Calculate the percentage columns for Ambig and mislab
entity_stats['Ambig_percentage'] = (entity_stats['AMBIG'] / entity_stats['fq']) * 100
entity_stats['Mislab_percentage'] = (entity_stats['MISLAB'] / entity_stats['fq']) * 100
# Calculate the percentage of valid entities
entity_stats['Valid_count'] = (entity_stats['fq'] - entity_stats['AMBIG'] - entity_stats['MISLAB'])

entity_stats['Valid_percentage'] = (entity_stats['Valid_count'] / entity_stats['fq']) * 100
entity_stats

### Graph debug

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Create figure and axes objects properly
fig, ax = plt.subplots(figsize=(12, 8), constrained_layout=True)

# Get the entity classes and percentages
classes = entity_stats['named_entity_class']
valid_percentages = entity_stats['Valid_percentage']
mislab_percentages = entity_stats['Mislab_percentage']
ambig_percentages = entity_stats['Ambig_percentage']

# Create positions for the bars
y_pos = np.arange(len(classes))
bar_height = 0.6

# Color scheme for consistent visualization
colors = {
    'valid': '#72b58e',    # Green for valid entities
    'mislab': '#f8a07e',   # Orange for mislabeled
    'ambig': '#8ecae6'     # Blue for ambiguous
}

# Set logarithmic scale for x-axis
ax.set_xscale('log')

# Handle zero values (can't represent in log scale)
epsilon = 1e-10
valid_log = np.array([max(v, epsilon) for v in valid_percentages])
mislab_log = np.array([max(m, epsilon) for m in mislab_percentages])
ambig_log = np.array([max(a, epsilon) for a in ambig_percentages])

# Plot stacked bars using the axis object
ax.barh(y_pos, valid_log, bar_height, color=colors['valid'], label='Valid')
ax.barh(y_pos, mislab_log, bar_height, left=valid_log, color=colors['mislab'], label='Mislabeled')
ax.barh(y_pos, ambig_log, bar_height, left=valid_log + mislab_log, color=colors['ambig'], label='Ambiguous')

# Add percentage labels with actual values
for i, (valid, mislab, ambig) in enumerate(zip(valid_percentages, mislab_percentages, ambig_percentages)):
    # Position labels appropriately in log space
    if valid > 0:
        log_pos = np.log10(valid/2 + epsilon)
        ax.text(10**log_pos, i, f'{valid:.1f}%', ha='center', va='center',
                fontweight='bold', color='black' if valid > 20 else 'white')

    if mislab > 0:
        log_pos = np.log10(valid + mislab/2 + epsilon)
        ax.text(10**log_pos, i, f'{mislab:.3f}%', ha='center', va='center',
                fontweight='bold', color='black')

    if ambig > 0:
        log_pos = np.log10(valid + mislab + ambig/2 + epsilon)
        ax.text(10**log_pos, i, f'{ambig:.3f}%', ha='center', va='center',
                fontweight='bold', color='black')

# Customize the plot
ax.set_yticks(y_pos)
ax.set_yticklabels(classes)
ax.set_xlabel('Percentage (%) - Log Scale')
ax.set_ylabel('Named Entity Class')
ax.set_title('Entity Classification Analysis by Entity Type (Log Scale)', fontsize=14)
ax.legend(loc='upper right')

# Create custom x-ticks for better log scale readability
log_ticks = [0.001, 0.01, 0.1, 1, 10, 100]
ax.set_xticks(log_ticks)
ax.set_xticklabels([f'{x}%' for x in log_ticks])
ax.grid(True, axis='x', linestyle='--', alpha=0.7)

# Add annotation explaining log scale
fig.text(0.5, 0.01,
         "Note: Logarithmic scale used to show both large and small percentages",
         ha="center", fontsize=10, style='italic')

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Create figure with two subplots
fig, (ax_percentage, ax_counts) = plt.subplots(1, 2, figsize=(18, 8), constrained_layout=True)

# Get the entity classes, percentages and counts
classes = entity_stats['named_entity_class']
valid_percentages = entity_stats['Valid_percentage']
mislab_percentages = entity_stats['Mislab_percentage']
ambig_percentages = entity_stats['Ambig_percentage']

# Calculate actual counts from percentages and total frequency
total_counts = entity_stats['fq']
valid_counts = total_counts * valid_percentages / 100
mislab_counts = total_counts * mislab_percentages / 100
ambig_counts = total_counts * ambig_percentages / 100

# Create positions for the bars
y_pos = np.arange(len(classes))
bar_height = 0.6

# Color scheme for consistent visualization
colors = {
    'valid': '#72b58e',    # Green for valid entities
    'mislab': '#f8a07e',   # Orange for mislabeled
    'ambig': '#8ecae6',     # Blue for ambiguous
}

# ---- LEFT SUBPLOT: PERCENTAGES (LOG SCALE) ----
ax_percentage.set_xscale('log')

# Handle zero values (can't represent in log scale)
epsilon = 1e-10
valid_log = np.array([max(v, epsilon) for v in valid_percentages])
mislab_log = np.array([max(m, epsilon) for m in mislab_percentages])
ambig_log = np.array([max(a, epsilon) for a in ambig_percentages])

# Plot stacked percentage bars
ax_percentage.barh(y_pos, valid_log, bar_height, color=colors['valid'], label='Valid')
ax_percentage.barh(y_pos, mislab_log, bar_height, left=valid_log, color=colors['mislab'], label='Mislabeled')
ax_percentage.barh(y_pos, ambig_log, bar_height, left=valid_log + mislab_log, color=colors['ambig'], label='Ambiguous')

# Add percentage labels
for i, (valid, mislab, ambig) in enumerate(zip(valid_percentages, mislab_percentages, ambig_percentages, strict=False)):
    # Position labels appropriately in log space
    if valid > 0:
        log_pos = np.log10(valid/2 + epsilon)
        ax_percentage.text(10**log_pos, i, f'{valid:.1f}%', ha='center', va='center',
                           fontweight='bold', color='black' if valid > 20 else 'white')

    if mislab > 0:
        log_pos = np.log10(valid + mislab/2 + epsilon)
        ax_percentage.text(10**log_pos, i, f'{mislab:.3f}%', ha='center', va='center',
                           fontweight='bold', color='black')

    if ambig > 0:
        log_pos = np.log10(valid + mislab + ambig/2 + epsilon)
        ax_percentage.text(10**log_pos, i, f'{ambig:.3f}%', ha='center', va='center',
                           fontweight='bold', color='black')

# Customize percentage subplot
ax_percentage.set_yticks(y_pos)
ax_percentage.set_yticklabels(classes)
ax_percentage.set_xlabel('Percentage (%) - Log Scale')
ax_percentage.set_ylabel('Named Entity Class')
ax_percentage.set_title('Relative Distribution (Percentages)', fontsize=12)
ax_percentage.legend(loc='upper right')

# Custom x-ticks for percentage subplot
log_percentage_ticks = [0.001, 0.01, 0.1, 1, 10, 100]
ax_percentage.set_xticks(log_percentage_ticks)
ax_percentage.set_xticklabels([f'{x}%' for x in log_percentage_ticks])
ax_percentage.grid(True, axis='x', linestyle='--', alpha=0.7)

# ---- RIGHT SUBPLOT: ACTUAL COUNTS (LOG SCALE) WITH REVERSED STACKING ----
ax_counts.set_xscale('log')

# Handle zero values for counts
ambig_counts_log = np.array([max(a, 1) for a in ambig_counts])
mislab_counts_log = np.array([max(m, 1) for m in mislab_counts])
valid_counts_log = np.array([max(v, 1) for v in valid_counts])

# Plot stacked count bars with reversed order (errors first)
ax_counts.barh(y_pos, ambig_counts_log, bar_height, color=colors['ambig'], label='Ambiguous')
ax_counts.barh(y_pos, mislab_counts_log, bar_height, left=ambig_counts_log, color=colors['mislab'], label='Mislabeled')
ax_counts.barh(y_pos, valid_counts_log, bar_height, left=ambig_counts_log + mislab_counts_log, color=colors['valid'], label='Valid')

# Add count labels
for i, (valid, mislab, ambig) in enumerate(zip(valid_counts, mislab_counts, ambig_counts, strict=False)):
    # Format large numbers with appropriate suffixes
    def format_count(count) -> None:
        if count == 0:
            return

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D

# Create figure with two subplots
fig, (ax_percentages, ax_counts) = plt.subplots(1, 2, figsize=(18, 7), constrained_layout=True)

# Adjust spacing to reduce vertical padding
plt.rcParams['figure.constrained_layout.h_pad'] = 0.05
plt.rcParams['figure.constrained_layout.w_pad'] = 0.05

# Get the entity classes, percentages and counts
classes = entity_stats['named_entity_class']
valid_percentages = entity_stats['Valid_percentage']
mislab_percentages = entity_stats['Mislab_percentage']
ambig_percentages = entity_stats['Ambig_percentage']

# Get the actual counts (either directly from entity_stats or computed from percentages)
total_counts = entity_stats['fq']
valid_counts = (valid_percentages * total_counts / 100).astype(int)
mislab_counts = (mislab_percentages * total_counts / 100).astype(int)
ambig_counts = (ambig_percentages * total_counts / 100).astype(int)

# Create positions for the bars
y_pos = np.arange(len(classes))
bar_height = 0.7  # Slightly increased bar height

# Color scheme for consistent visualization
colors = {
    'valid': '#72b58e',    # Green for valid entities
    'mislab': '#f8a07e',   # Orange for mislabeled
    'ambig': '#8ecae6',     # Blue for ambiguous
}

# ---- LEFT SUBPLOT: PERCENTAGES (LOG SCALE) ----
ax_percentages.set_xscale('log')

# Handle zero values (can't represent in log scale)
epsilon = 1e-10
valid_log = np.array([max(v, epsilon) for v in valid_percentages])
mislab_log = np.array([max(m, epsilon) for m in mislab_percentages])
ambig_log = np.array([max(a, epsilon) for a in ambig_percentages])

# Plot stacked percentage bars
valid_bars = ax_percentages.barh(y_pos, valid_log, bar_height, color=colors['valid'], label='Valid')
mislab_bars = ax_percentages.barh(y_pos, mislab_log, bar_height, left=valid_log, color=colors['mislab'], label='Mislabeled')
ambig_bars = ax_percentages.barh(y_pos, ambig_log, bar_height, left=valid_log + mislab_log, color=colors['ambig'], label='Ambiguous')

# Improved log-space positioning for text labels
for i, (valid, mislab, ambig) in enumerate(zip(valid_percentages, mislab_percentages, ambig_percentages, strict=False)):
    # Valid segment - proper log-space calculation
    if valid > 0.5:  # Only show if enough space
        log_mid_point = np.log10(epsilon + valid/2) if valid > 0 else 0
        ax_percentages.text(10**log_mid_point, i, f'{valid:.1f}%', ha='center', va='center',
                           fontweight='bold', color='black' if valid > 20 else 'white')

    # Mislabeled segment - proper log-space calculation
    if mislab > 0.01:
        # Find midpoint in original space, then convert to log space
        mid_point_orig = valid + mislab/2
        log_mid_point = np.log10(mid_point_orig)
        ax_percentages.text(10**log_mid_point, i, f'{mislab:.2f}%', ha='center', va='center',
                           fontweight='bold', color='black')

    # Ambiguous segment - proper log-space calculation
    if ambig > 0.01:
        # Find midpoint in original space, then convert to log space
        mid_point_orig = valid + mislab + ambig/2
        log_mid_point = np.log10(mid_point_orig)
        ax_percentages.text(10**log_mid_point, i, f'{ambig:.2f}%', ha='center', va='center',
                           fontweight='bold', color='black')

In [ ]:
# Customize percentage subplot
ax_percentages.set_yticks(y_pos)
ax_percentages.set_yticklabels(classes)
ax_percentages.set_xlabel('Percentage (%) - Log Scale')
ax_percentages.set_ylabel('Named Entity Class')
ax_percentages.set_title('A: Distribution by Percentage', fontsize=14)

# Custom x-ticks for percentage subplot
log_percentage_ticks = [0.001, 0.01, 0.1, 1, 10, 100]
ax_percentages.set_xticks(log_percentage_ticks)
ax_percentages.set_xticklabels([f'{x}%' for x in log_percentage_ticks])
ax_percentages.grid(True, axis='x', linestyle='--', alpha=0.7)

# ---- RIGHT SUBPLOT: ABSOLUTE COUNTS (LOG SCALE) WITH REVERSED STACKING ----
ax_counts.set_xscale('log')

# Handle zero values for counts
ambig_counts_log = np.array([max(a, 1) for a in ambig_counts])
mislab_counts_log = np.array([max(m, 1) for m in mislab_counts])
valid_counts_log = np.array([max(v, 1) for v in valid_counts])

# Plot stacked count bars with reversed order (errors first)
ambig_bars2 = ax_counts.barh(y_pos, ambig_counts_log, bar_height, color=colors['ambig'], label='Ambiguous')
mislab_bars2 = ax_counts.barh(y_pos, mislab_counts_log, bar_height, left=ambig_counts_log, color=colors['mislab'], label='Mislabeled')
valid_bars2 = ax_counts.barh(y_pos, valid_counts_log, bar_height, left=ambig_counts_log + mislab_counts_log, color=colors['valid'], label='Valid')

# Define function to format counts with appropriate suffixes
def format_count(count) -> str:
    if count == 0:
        return "0"
    elif count < 1000:
        return f"{count:,.0f}"
    elif count < 1000000:
        return f"{count/1000:.1f}K"
    else:
        return f"{count/1000000:.1f}M"

# Improved log-space positioning for count labels
for i, (ambig, mislab, valid) in enumerate(zip(ambig_counts, mislab_counts, valid_counts, strict=False)):
    # Ambiguous segment - properly calculate log-space midpoint
    if ambig > 10:
        log_mid_point = np.log10(ambig/2 + 1)
        ax_counts.text(10**log_mid_point, i, format_count(ambig), ha='center', va='center',
                      fontweight='bold', color='black')

    # Mislabeled segment - properly calculate log-space midpoint
    if mislab > 10:
        mid_point_orig = ambig + mislab/2
        log_mid_point = np.log10(mid_point_orig)
        ax_counts.text(10**log_mid_point, i, format_count(mislab), ha='center', va='center',
                      fontweight='bold', color='black')

    # Valid segment - properly calculate log-space midpoint
    if valid > 100:
        mid_point_orig = ambig + mislab + valid/2
        log_mid_point = np.log10(mid_point_orig)
        ax_counts.text(10**log_mid_point, i, format_count(valid), ha='center', va='center',
                      fontweight='bold', color='white' if valid > 1000000 else 'black')


In [ ]:
# Customize counts subplot
ax_counts.set_yticks(y_pos)
ax_counts.set_yticklabels([])  # Hide y-labels on right plot
ax_counts.set_xlabel('Entity Count (log scale)')
ax_counts.set_title('B: Distribution by Count (Errors First)', fontsize=14)

# Generate appropriate log ticks based on data range
max_count = max(total_counts)
magnitude = int(np.log10(max_count)) + 1
log_count_ticks = [10**i for i in range(0, magnitude)]
ax_counts.set_xticks(log_count_ticks)
ax_counts.set_xticklabels([format_count(x) for x in log_count_ticks])
ax_counts.grid(True, axis='x', linestyle='--', alpha=0.7)

for tl in ax.get_yticklabels():
    txt = tl.get_text()
    if txt == 'PNM':
        tl.set_backgroundcolor('C3')


# Create a single shared legend at the top
legend_elements = [
    Line2D([0], [0], color=colors['valid'], lw=8, label='Valid'),
    Line2D([0], [0], color=colors['mislab'], lw=8, label='Mislabeled'),
    Line2D([0], [0], color=colors['ambig'], lw=8, label='Ambiguous'),
]
fig.legend(handles=legend_elements, loc='upper center', ncol=3, frameon=True,
           bbox_to_anchor=(0.5, 0.9), fontsize=13)

# Main title - positioned to make room for legend
fig.suptitle('Entity Classification Analysis', fontsize=16, y=0.99)

# Add annotation explaining the visualization
fig.text(0.5, 0.01,
         "Note: Both plots use logarithmic scale. A: percentages with valid first. B: counts with errors first.",
         ha="center", fontsize=10, style='italic')

# Fine-tune layout
plt.tight_layout(rect=[0, 0.03, 1, 0.90])  # Make room for shared legend and annotation
plt.subplots_adjust(wspace=0.05)  # Reduce space between subplots


In [ ]:
plt.show()

### Graph run

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D

# Create figure with two subplots
fig, (ax_percentages, ax_counts) = plt.subplots(1, 2, figsize=(18, 7), constrained_layout=True)

# Adjust spacing to reduce vertical padding
plt.rcParams['figure.constrained_layout.h_pad'] = 0.05
plt.rcParams['figure.constrained_layout.w_pad'] = 0.05

# Get the entity classes, percentages and counts
classes = entity_stats['named_entity_class']
valid_percentages = entity_stats['Valid_percentage']
mislab_percentages = entity_stats['Mislab_percentage']
ambig_percentages = entity_stats['Ambig_percentage']

# Get the actual counts (either directly from entity_stats or computed from percentages)
total_counts = entity_stats['fq']
valid_counts = (valid_percentages * total_counts / 100).astype(int)
mislab_counts = (mislab_percentages * total_counts / 100).astype(int)
ambig_counts = (ambig_percentages * total_counts / 100).astype(int)



# Create positions for the bars
y_pos = np.arange(len(classes))
bar_height = 0.7  # Slightly increased bar height

# Color scheme for consistent visualization
colors = {
    'valid': '#72b58e',    # Green for valid entities
    'mislab': '#f8a07e',   # Orange for mislabeled
    'ambig': '#8ecae6',     # Blue for ambiguous
}

# ---- LEFT SUBPLOT: PERCENTAGES (LOG SCALE) ----
ax_percentages.set_xscale('log')

# Handle zero values (can't represent in log scale)
epsilon = 1e-10
valid_log = np.array([max(v, epsilon) for v in valid_percentages])
mislab_log = np.array([max(m, epsilon) for m in mislab_percentages])
ambig_log = np.array([max(a, epsilon) for a in ambig_percentages])

# Plot stacked percentage bars
valid_bars = ax_percentages.barh(y_pos, valid_log, bar_height, color=colors['valid'], label='Valid')
mislab_bars = ax_percentages.barh(y_pos, mislab_log, bar_height, left=valid_log, color=colors['mislab'], label='Mislabeled')
ambig_bars = ax_percentages.barh(y_pos, ambig_log, bar_height, left=valid_log + mislab_log, color=colors['ambig'], label='Ambiguous')

# Improved log-space positioning for text labels
for i, (valid, mislab, ambig) in enumerate(zip(valid_percentages, mislab_percentages, ambig_percentages, strict=False)):
    # Valid segment - proper log-space calculation
    if valid > 0.5:  # Only show if enough space
        log_mid_point = np.log10(epsilon + valid/2) if valid > 0 else 0
        ax_percentages.text(10**log_mid_point, i, f'{valid:.1f}%', ha='center', va='center',
                        fontweight='bold', color='black' if valid > 20 else 'white')

    # Mislabeled segment - proper log-space calculation
    if mislab > 0.01:
        # Find midpoint in original space, then convert to log space
        mid_point_orig = valid + mislab/2
        log_mid_point = np.log10(mid_point_orig)
        ax_percentages.text(10**log_mid_point, i, f'{mislab:.2f}%', ha='center', va='center',
                        fontweight='bold', color='black')

    # Ambiguous segment - proper log-space calculation
    if ambig > 0.01:
        # Find midpoint in original space, then convert to log space
        mid_point_orig = valid + mislab + ambig/2
        log_mid_point = np.log10(mid_point_orig)
        ax_percentages.text(10**log_mid_point, i, f'{ambig:.2f}%', ha='center', va='center',
                        fontweight='bold', color='black')

# Customize percentage subplot
ax_percentages.set_yticks(y_pos)
ax_percentages.set_yticklabels(classes)
ax_percentages.set_xlabel('Percentage (%) - Log Scale')
ax_percentages.set_ylabel('Named Entity Class')
ax_percentages.set_title('A: Distribution by Percentage', fontsize=14)

# Custom x-ticks for percentage subplot
log_percentage_ticks = [0.001, 0.01, 0.1, 1, 10, 100]
ax_percentages.set_xticks(log_percentage_ticks)
ax_percentages.set_xticklabels([f'{x}%' for x in log_percentage_ticks])
ax_percentages.grid(True, axis='x', linestyle='--', alpha=0.7)

# ---- RIGHT SUBPLOT: ABSOLUTE COUNTS (LOG SCALE) WITH REVERSED STACKING ----
ax_counts.set_xscale('log')

# Handle zero values for counts
ambig_counts_log = np.array([max(a, 1) for a in ambig_counts])
mislab_counts_log = np.array([max(m, 1) for m in mislab_counts])
valid_counts_log = np.array([max(v, 1) for v in valid_counts])

# Plot stacked count bars with reversed order (errors first)
ambig_bars2 = ax_counts.barh(y_pos, ambig_counts_log, bar_height, color=colors['ambig'], label='Ambiguous')
mislab_bars2 = ax_counts.barh(y_pos, mislab_counts_log, bar_height, left=ambig_counts_log, color=colors['mislab'], label='Mislabeled')
valid_bars2 = ax_counts.barh(y_pos, valid_counts_log, bar_height, left=ambig_counts_log + mislab_counts_log, color=colors['valid'], label='Valid')

# Define function to format counts with appropriate suffixes
def format_count(count) -> str:
    if count == 0:
        return "0"
    elif count < 1000:
        return f"{count:,.0f}"
    elif count < 1000000:
        return f"{count/1000:.1f}K"
    else:
        return f"{count/1000000:.1f}M"

# Improved log-space positioning for count labels
for i, (ambig, mislab, valid) in enumerate(zip(ambig_counts, mislab_counts, valid_counts, strict=False)):
    # Ambiguous segment - properly calculate log-space midpoint
    if ambig > 10:
        log_mid_point = np.log10(ambig/2 + 1)
        ax_counts.text(10**log_mid_point, i, format_count(ambig), ha='center', va='center',
                    fontweight='bold', color='black')

    # Mislabeled segment - properly calculate log-space midpoint
    if mislab > 10:
        mid_point_orig = ambig + mislab/2
        log_mid_point = np.log10(mid_point_orig)
        ax_counts.text(10**log_mid_point, i, format_count(mislab), ha='center', va='center',
                    fontweight='bold', color='black')

    # Valid segment - properly calculate log-space midpoint
    if valid > 100:
        mid_point_orig = ambig + mislab + valid/2
        log_mid_point = np.log10(mid_point_orig)
        ax_counts.text(10**log_mid_point, i, format_count(valid), ha='center', va='center',
                    fontweight='bold', color='white' if valid > 1000000 else 'black')

# Customize counts subplot
ax_counts.set_yticks(y_pos)
ax_counts.set_yticklabels([])  # Hide y-labels on right plot
ax_counts.set_xlabel('Entity Count (log scale)')
ax_counts.set_title('B: Distribution by Count (Errors First)', fontsize=14)

# Generate appropriate log ticks based on data range
max_count = max(total_counts)
magnitude = int(np.log10(max_count)) + 1
log_count_ticks = [10**i for i in range(0, magnitude)]
ax_counts.set_xticks(log_count_ticks)
ax_counts.set_xticklabels([format_count(x) for x in log_count_ticks])
ax_counts.grid(True, axis='x', linestyle='--', alpha=0.7)

for tl in ax_percentages.get_yticklabels():
    txt = tl.get_text()
    if txt == 'PNM':
        tl.set_backgroundcolor('')

# Create a single shared legend at the top
legend_elements = [
    Line2D([0], [0], color=colors['valid'], lw=8, label='Valid'),
    Line2D([0], [0], color=colors['mislab'], lw=8, label='Mislabeled'),
    Line2D([0], [0], color=colors['ambig'], lw=8, label='Ambiguous'),
]
fig.legend(handles=legend_elements, loc='upper center', ncol=3, frameon=True,
        bbox_to_anchor=(0.5, 0.9), fontsize=13)

# Main title - positioned to make room for legend
fig.suptitle('Entity Classification Analysis', fontsize=16, y=0.99)

# Add annotation explaining the visualization
fig.text(0.5, 0.01,
        "Note: Both plots use logarithmic scale. A: percentages with valid first. B: counts with errors first.",
        ha="center", fontsize=10, style='italic')

# Fine-tune layout
plt.tight_layout(rect=[0, 0.03, 1, 0.90])  # Make room for shared legend and annotation
plt.subplots_adjust(wspace=0.05)  # Reduce space between subplots

plt.show()

In [ ]:
import easyner.database.sqlite_backend.statistics.stacked_bar_plots

In [ ]:
from stacked_bar_plots import plot_stacked_bar

plot_stacked_bar(entity_stats)